In [58]:
import pandas as pd
import numpy as np
import random
import time
import json
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel 
from transformers import logging

In [59]:
behaviors = pd.read_csv('train/train_behaviors.tsv', delimiter='\t', index_col=0, header=None)
behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']
display(behaviors)

,user,time,clicked_news,impressions
0,,,,
0,U1349561,11/11/2019 9:31:08 AM,N410559 N109405 N79284 N812877 N311012 N362483...,N383574-0 N727666-0 N169045-0 N846428-0 N70382...
1,U2788121,11/9/2019 9:13:19 AM,N642952 N253717 N857922 N684266 N291776 N12107...,N184823-0 N107900-0 N29500-0 N122187-0 N487132...
2,U686145,11/12/2019 6:21:28 AM,N900496 N118253 N510477 N167498 N693772,N499466-0 N665940-0 N394508-1 N386423-0 N39675...
3,U2794941,11/13/2019 9:30:05 AM,N417895,N96322-0 N909778-0 N656987-0 N594694-0 N306259...
4,U1838845,11/10/2019 5:03:16 AM,N187833 N272183 N482344 N65242 N211696 N194448...,N27904-0 N731054-0 N281766-0 N177459-0 N768567...
...,...,...,...,...
285292,U2741389,11/12/2019 5:49:13 AM,N9067 N895456 N560877 N28282 N836782 N675040 N...,N499753-0 N427450-0 N429683-0 N100613-0 N57620...
285293,U1710149,11/12/2019 2:56:30 AM,N285350 N104848,N678008-0 N362091-0 N639662-0 N308947-0 N57902...
285294,U1519153,11/13/2019 5:12:48 AM,N649378 N609730 N365374 N695774 N647131 N59538...,N717138-1 N785087-0 N642007-0 N408928-0 N52696...


In [60]:
behaviors = pd.read_csv('train/train_behaviors.tsv', delimiter='\t', index_col=0, header=None)
behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']
behaviors = behaviors.sample(frac=1, random_state=42)
train_behaviors = behaviors[:int(len(behaviors) * 0.8)]
valid_behaviors = behaviors[int(len(behaviors) * 0.8):]

news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}
embedding = {}
f = open("train/train_entity_embedding.vec")
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embedding[word] = coefs
f.close()

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Test Dataset

In [61]:
test_behaviors = pd.read_csv('test/test_behaviors.tsv', delimiter='\t', index_col=0, header=None)
test_behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']

test_news = pd.read_csv('test/test_news.tsv', delimiter='\t', header=None)
test_news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
test_news_dict = {data['news_id']: data.iloc[1:] for _, data in test_news.iterrows()}
test_embedding = {}
f = open("test/test_entity_embedding.vec")
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    test_embedding[word] = coefs
f.close()

In [62]:
class RecommendationDataset(Dataset):
    def __init__(self, behaviors, news_dict, embedding, tokenizer, mode):
        self.behaviors = behaviors
        self.news_dict = news_dict
        self.embedding = embedding
        self.tokenizer = tokenizer
        self.mode = mode
        
    def __len__(self):
        return len(self.behaviors)

    def padding(self, item, size):
        item = torch.stack(item)[:size]
        if item.size(0) < size:
            padding_length = size - item.size(0)
            padding_item = torch.zeros((padding_length, *item.shape[1:]))
            item = torch.cat((item, padding_item), dim=0)
        return item
    
    def extract_entities(self, news):
        _, _, _, _, _, title_entities, abstract_entities = self.news_dict[news]
        
        title_entities = "[]" if isinstance(title_entities, float) else title_entities
        abstract_entities = "[]" if isinstance(abstract_entities, float) else abstract_entities
        
        vector = []
        entities = json.loads(title_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]
        entities = json.loads(abstract_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]

        if len(vector) == 0:
            vector = torch.zeros((30, 100))
        else:
            vector = self.padding(vector, 30)
                
        return vector

    def extract_text(self, news):
        category, subcategory, title, abstract, _, _, _ = self.news_dict[news]
        
        title = "" if isinstance(title, float) else title
        abstract = "" if isinstance(abstract, float) else abstract
        encoding = self.tokenizer(
            text = category + " " + subcategory,
            text_pair = title + " " + abstract,
            return_tensors = "pt",
            padding = "max_length",
            truncation = True,
            max_length = 200
        )
        return encoding
    
    def __getitem__(self, index):
        _, _, clicked_news, impressions = self.behaviors.iloc[index]
        history_vectors = []
        history_encodings = []
        clicked_news = clicked_news.split()
        for news in clicked_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            history_vectors.append(vector)
            history_encodings.append(encoding)
        history_vectors = self.padding(history_vectors, 20)
        history_ids = [encoding.input_ids[0, :] for encoding in history_encodings]
        history_ids = self.padding(history_ids, 20)
        history_type = [encoding.token_type_ids[0, :] for encoding in history_encodings]
        history_type = self.padding(history_type, 20)
        history_mask = [encoding.attention_mask[0, :] for encoding in history_encodings]
        history_mask = self.padding(history_mask, 20)

        recommen_vectors = []
        recommen_encodings = []
        impressions = impressions.split()
        if(self.mode == "Train"):
          labels = [int(impression.split('-')[1]) for impression in impressions]
        impression_news = [impression.split('-')[0] for impression in impressions]
        for news in impression_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            recommen_vectors.append(vector)
            recommen_encodings.append(encoding)
        recommen_ids = [encoding.input_ids for encoding in recommen_encodings]
        recommen_ids = torch.cat(recommen_ids)
        recommen_type = [encoding.token_type_ids for encoding in recommen_encodings]
        recommen_type = torch.cat(recommen_type)
        recommen_mask = [encoding.attention_mask for encoding in recommen_encodings]
        recommen_mask = torch.cat(recommen_mask)

        packed = {
            'history_vectors':  history_vectors,
            'history_ids':      history_ids,
            'history_type':     history_type,
            'history_mask':     history_mask,
            'recommen_vectors': recommen_vectors,
            'recommen_ids':     recommen_ids,
            'recommen_type':    recommen_type,
            'recommen_mask':    recommen_mask,
        }
        if(self.mode == "Train"):
          return packed, labels
        elif(self.mode == "Test"):
          return packed
train_dataset = RecommendationDataset(train_behaviors, news_dict, embedding, tokenizer, "Train")
valid_dataset = RecommendationDataset(valid_behaviors, news_dict, embedding, tokenizer, "Train")
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

In [70]:
# for idx, (item, label) in enumerate(train_loader):
#   print(item['history_vectors'].shape)
#   print(item['history_ids'].shape)
#   print(item['history_type'].shape)
#   print(item['history_mask'].shape)
#   break


torch.Size([16, 20, 30, 100])
torch.Size([16, 20, 200])
torch.Size([16, 20, 200])
torch.Size([16, 20, 200])


# Test DataLoader

In [49]:
test_dataset = RecommendationDataset(test_behaviors, test_news_dict, test_embedding, tokenizer, "Test")
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [52]:
class RecommendationModel(nn.Module):
    def __init__(self):
        super(RecommendationModel, self).__init__()
        
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.linear = nn.Linear(200, 15)
    
    def forward(self, history_vectors, history_ids, history_type, history_mask, \
        recommen_vectors, recommen_ids, recommen_type, recommen_mask):
        return self.linear(history_ids[:, 0, :])

In [56]:
device = torch.device("cuda")
num_epoch = 5
show_freq = 10

model = RecommendationModel()
model = model.to(device)

loss = nn.MultiLabelSoftMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

best_valid_auc = 0.0
for epoch in range(num_epoch):
    epoch_start_time = time.time()
    train_loss, valid_loss = 0.0, 0.0
    train_count, valid_count = 0.0, 0.0
    train_true, valid_true = [], []
    train_pred, valid_pred = [], []

    model.train()
    for i, (packed, labels) in enumerate(train_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
            
        outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        batch_loss.backward()
        optimizer.step()
        model.zero_grad()

        train_loss += batch_loss.item()
        train_count += labels.shape[0] * labels.shape[1]
        train_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        train_pred += outputs.reshape(-1).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(train_loader):
            train_auc = roc_auc_score(train_true, train_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(train_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(train_auc, train_loss/train_count)
            )

                
    model.eval()
    for i, (packed, labels) in enumerate(valid_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        
        valid_loss += batch_loss.item()
        valid_count += labels.shape[0] * labels.shape[1]
        valid_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        valid_pred += outputs.reshape(-1).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(valid_loader):
            valid_auc = roc_auc_score(valid_true, valid_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(valid_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(valid_auc, valid_loss/valid_count)
            )
            
    if best_valid_auc < valid_auc:
      best_valid_auc = valid_auc
      torch.save(model.state_dict(), 'best_weight.pth')
        
print(f"Best Validation AUC: {best_valid_auc}")


RuntimeError: ignored